In [3]:
import sys
import yaml
from pathlib import Path
import pandas as pd

# Add project root to path
project_root = Path.cwd().parent.parent.parent
sys.path.insert(0, str(project_root / "src"))

from bonn_thesis.openai_processing.isco_fine_tune_manager import (
    upload_file,
    create_fine_tune_job,
    get_job_status,
    list_fine_tune_jobs,
    save_job_info,
)
from bonn_thesis.config import OCCUPATION_DATA_BLD, SRC
from openai import OpenAI

## 1. Load Configuration and Metadata

In [4]:
# Initialize OpenAI client
client = OpenAI()  # Will use OPENAI_API_KEY environment variable

# Load fine-tune configuration
config_path = SRC / "openai_processing" / "configs" / "fine_tune" / "isco_classifier_04.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

# Load metadata to get file paths
metadata_path = OCCUPATION_DATA_BLD / "fine_tune_metadata.csv"
metadata = pd.read_csv(metadata_path)

training_file = Path(metadata["training_file"].iloc[0])
validation_file_str = metadata["validation_file"].iloc[0]
validation_file = Path(validation_file_str) if pd.notna(validation_file_str) else None

print(f"Configuration: {config['fine_tune_id']}")
print(f"Base model: {config['model']}")
print(f"Epochs: {config['n_epochs']}")
print(f"\nTraining examples: {metadata['n_training_examples'].iloc[0]}")
print(f"Validation examples: {metadata['n_validation_examples'].iloc[0]}")
print(f"\nEstimated training cost: ${metadata['training_cost'].iloc[0]:.4f}")
print(f"Estimated cost per 1k predictions: ${metadata['inference_cost_per_1k'].iloc[0]:.4f}")

Configuration: isco_classifier_04
Base model: gpt-4o-mini-2024-07-18
Epochs: 3

Training examples: 7496
Validation examples: 0

Estimated training cost: $3.3923
Estimated cost per 1k predictions: $0.0110


## 2. Upload Files to OpenAI

In [6]:
# Upload training file
print("Uploading training file...")
training_file_id = upload_file(client, training_file)
print(f"✓ Training file uploaded: {training_file_id}")

# Upload validation file if it exists and is not empty
validation_file_id = None
if validation_file and validation_file.exists():
    # Check if file has content (more than just header)
    try:
        val_df = pd.read_json(validation_file, lines=True)
        if len(val_df) > 0:
            print("\nUploading validation file...")
            validation_file_id = upload_file(client, validation_file)
            print(f"✓ Validation file uploaded: {validation_file_id}")
        else:
            print("\n⚠️ Validation file is empty, skipping validation data")
    except (ValueError, pd.errors.EmptyDataError):
        print("\n⚠️ Validation file is empty or invalid, skipping validation data")
else:
    print("\n⚠️ No validation file found, training without validation data")

Uploading training file...
✓ Training file uploaded: file-Djb9kYfBeJue9UW8jv86tB

⚠️ Validation file is empty, skipping validation data


## 3. Create Fine-Tuning Job

In [7]:
# Create the fine-tuning job
print("Creating fine-tuning job...")
job_info = create_fine_tune_job(
    client=client,
    training_file_id=training_file_id,
    config=config,
    validation_file_id=validation_file_id,
)

print("\n✓ Fine-tuning job created!")
print(f"Job ID: {job_info['job_id']}")
print(f"Status: {job_info['status']}")
print(f"Base model: {job_info['model']}")

# Save job info to metadata
save_job_info(job_info, metadata_path)

# Save job ID for later use
job_id = job_info["job_id"]

Creating fine-tuning job...

✓ Fine-tuning job created!
Job ID: ftjob-4BRPlKiHU3OFIGXnaA1qM5o3
Status: validating_files
Base model: gpt-4o-mini-2024-07-18


## 4. Check Job Status

Run this cell periodically to check the status of your fine-tuning job.

In [10]:
# Check status
status_info = get_job_status(client, job_id)

print(f"Job ID: {status_info['job_id']}")
print(f"Status: {status_info['status']}")
print(f"Fine-tuned model: {status_info.get('fine_tuned_model', 'Not available yet')}")
print(f"Trained tokens: {status_info.get('trained_tokens', 'Not available yet')}")

if status_info.get('error'):
    print(f"\n❌ Error: {status_info['error']}")
elif status_info['status'] == 'succeeded':
    print("\n✓ Fine-tuning completed successfully!")
    # Update metadata with final status
    save_job_info(status_info, metadata_path)
elif status_info['status'] in ['validating', 'queued', 'running']:
    print("\n⏳ Job is still in progress. Check back later.")

Job ID: ftjob-4BRPlKiHU3OFIGXnaA1qM5o3
Status: succeeded
Fine-tuned model: ft:gpt-4o-mini-2024-07-18:personal:isco-classifier-v04:Cxjj4e3J
Trained tokens: 1081356

❌ Error: Error(code=None, message=None, param=None)


## 5. List All Fine-Tuning Jobs

View all recent fine-tuning jobs.

In [11]:
# List all recent jobs
jobs = list_fine_tune_jobs(client, limit=10)

print("Recent fine-tuning jobs:\n")
for i, job in enumerate(jobs, 1):
    print(f"{i}. Job ID: {job['job_id']}")
    print(f"   Status: {job['status']}")
    print(f"   Base Model: {job['model']}")
    print(f"   Fine-tuned Model: {job.get('fine_tuned_model', 'N/A')}")
    print()

Recent fine-tuning jobs:

1. Job ID: ftjob-4BRPlKiHU3OFIGXnaA1qM5o3
   Status: succeeded
   Base Model: gpt-4o-mini-2024-07-18
   Fine-tuned Model: ft:gpt-4o-mini-2024-07-18:personal:isco-classifier-v04:Cxjj4e3J

2. Job ID: ftjob-wAczhnf8DXGCxpCY3OM1CRS5
   Status: succeeded
   Base Model: gpt-4o-mini-2024-07-18
   Fine-tuned Model: ft:gpt-4o-mini-2024-07-18:personal:isco-classifier-v2:Ctz8Owre

3. Job ID: ftjob-pBGx57DZMK4s1pI9mHXCdhoj
   Status: cancelled
   Base Model: gpt-4o-mini-2024-07-18
   Fine-tuned Model: None

4. Job ID: ftjob-oL5la31RAwV2t6rl9OM7oHa7
   Status: succeeded
   Base Model: gpt-4o-mini-2024-07-18
   Fine-tuned Model: ft:gpt-4o-mini-2024-07-18:personal:isco-classifier-v2:CtkUaNss

5. Job ID: ftjob-v1f37jSiJ6Ua2KUR6xyvViYK
   Status: succeeded
   Base Model: gpt-4.1-nano-2025-04-14
   Fine-tuned Model: ft:gpt-4.1-nano-2025-04-14:personal:isco-classifier-v1:Ctfry0IY



## 6. Test the Fine-Tuned Model

Once the job is complete, test the model on sample job titles.

In [12]:
# Get the fine-tuned model name from the job status
status_info = get_job_status(client, job_id)
fine_tuned_model = status_info.get('fine_tuned_model')

if not fine_tuned_model:
    print("❌ Fine-tuned model not available yet. Job status:", status_info['status'])
else:
    print(f"Testing model: {fine_tuned_model}\n")
    
    # Test job titles
    test_jobs = [
        "Software Engineer",
        "Data Scientist",
        "Registered Nurse",
        "Primary School Teacher",
        "Sales Manager",
        "Chef",
    ]
    
    system_message = config['system_message']
    
    print(f"{'Job Title':<40} {'ISCO Code'}")
    print("-" * 55)
    
    for job_title in test_jobs:
        response = client.chat.completions.create(
            model=fine_tuned_model,
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": job_title},
            ],
            temperature=0,
        )
        isco_code = response.choices[0].message.content
        print(f"{job_title:<40} {isco_code}")

Testing model: ft:gpt-4o-mini-2024-07-18:personal:isco-classifier-v04:Cxjj4e3J

Job Title                                ISCO Code
-------------------------------------------------------
Software Engineer                        251
Data Scientist                           252
Registered Nurse                         222
Primary School Teacher                   234
Sales Manager                            122
Chef                                     343
